# Lorenz-1960: 30-window long-horizon PINN

Run on **Google Colab** with a T4 GPU if available. This is a long-horizon [0,15] experiment, not a closed-orbit run. The notebook pins the code commit, verifies the effective points in all 30 windows, runs the project numerical tests, then trains, evaluates, and renders the final figures. Reference values are used only for evaluation and endpoint reports.

**Persistent storage:** the repository and `runs/causal-window/long-horizon-30/seed0/history/progress.pt` live in `MyDrive/lorenz1960-window30`. An interrupted run resumes when you rerun these cells on the same Drive path and same device model. Do not delete that folder during training. The final cell makes a compact ZIP for copying the checkpoint, configuration, CSVs, report, and figures back into the branch.


In [ ]:
from pathlib import Path
import os, subprocess, sys, urllib.request
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'feat/causal-window-30'
CODE_COMMIT = '57172f6e00f6b537ca3b78f27f5dc21b271b7f21'
REMOTE = 'https://github.com/ihmorol/lorenz1960-pinn.git'
PROJECT = Path('/content/drive/MyDrive/lorenz1960-window30')
REPO = PROJECT / 'repo'
PROJECT.mkdir(parents=True, exist_ok=True)
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', '--depth=3',
                    '--branch', BRANCH, REMOTE, str(REPO)], check=True)
has_commit = subprocess.run(['git', '-C', str(REPO), 'cat-file', '-e',
                             f'{CODE_COMMIT}^{{commit}}'], capture_output=True).returncode == 0
if not has_commit:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', CODE_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO), 'sparse-checkout', 'init', '--cone'], check=True)
subprocess.run(['git', '-C', str(REPO), 'sparse-checkout', 'set', 'src', 'colab'], check=True)
subprocess.run(['git', '-C', str(REPO), 'switch', '-C', BRANCH, CODE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == CODE_COMMIT
# Sparse checkout avoids downloading hundreds of MB of historical films and figures.
for name in ('run_causal_window_30.py', 'requirements-colab-window30.txt'):
    path = REPO / name
    if not path.exists():
        url = f'https://raw.githubusercontent.com/ihmorol/lorenz1960-pinn/{CODE_COMMIT}/{name}'
        urllib.request.urlretrieve(url, path)
old = REPO / 'runs/4x60_f64_unit_win27_causal_warm/history/pinn.pt'
old.parent.mkdir(parents=True, exist_ok=True)
if not old.exists():
    url = f'https://raw.githubusercontent.com/ihmorol/lorenz1960-pinn/{CODE_COMMIT}/runs/4x60_f64_unit_win27_causal_warm/history/pinn.pt'
    urllib.request.urlretrieve(url, old)
print('Exact branch and code commit:', BRANCH, CODE_COMMIT)
print('Persistent repository:', REPO)
print('Historical checkpoint bytes:', old.stat().st_size)


In [ ]:
import importlib.metadata as metadata, json, platform, torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(REPO / 'requirements-colab-window30.txt')], check=True)
import numpy, scipy, pandas, matplotlib, seaborn, plotly, pytest
assert torch.__version__.split('+')[0] == '2.11.0'
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before the full run.'
hardware = {'python': platform.python_version(), 'torch': torch.__version__,
            'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
            'numpy': numpy.__version__, 'scipy': scipy.__version__,
            'pandas': pandas.__version__, 'matplotlib': matplotlib.__version__,
            'seaborn': seaborn.__version__, 'plotly': plotly.__version__,
            'pytest': pytest.__version__}
print(json.dumps(hardware, indent=2))


In [ ]:
subprocess.run([sys.executable, 'run_causal_window_30.py', '--verify-only'], cwd=REPO, check=True)
numerical_tests = ['test_hard_ic_exact', 'test_residual_zero_on_truth',
                   'test_collocation_samplers_cover_the_span', 'test_float64_trains_end_to_end',
                   'test_causal_weights_gate_later_times', 'test_split_windows_tile_the_span',
                   'test_windowed_pinn_routes_and_is_continuous',
                   'test_eps_advances_only_when_all_weights_exceed_delta',
                   'test_warm_start_copies_previous_window_weights',
                   'test_adam_checkpoint_resumes', 'test_windowed_resume_skips_saved_windows']
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                *[f'src/pinn/test_pinn.py::{name}' for name in numerical_tests]],
               cwd=REPO, check=True)
print('Numerical tests and 30-window point-count preflight passed in Colab.')


In [ ]:
# This cell can take a long time. On a disconnect, reconnect to the same T4 runtime
# type and rerun the notebook; progress.pt resumes within the active window.
subprocess.run([sys.executable, 'run_causal_window_30.py'], cwd=REPO, check=True)
print((REPO / 'runs/causal-window/long-horizon-30/seed0/report.md').read_text())


In [ ]:
# Create a compact export; no historical run files are included.
import shutil
run = REPO / 'runs/causal-window/long-horizon-30/seed0'
assert (run / 'metadata.json').exists(), 'The full run and evaluation have not finished.'
archive = shutil.make_archive(str(PROJECT / 'causal-window-30-results'), 'zip', root_dir=run)
print('Persistent ZIP:', archive)
print('Files:', *sorted(str(p.relative_to(run)) for p in run.rglob('*') if p.is_file()), sep='\n')
